# Multi-Domain Log Anomaly Detection with Data Augmentation

여러 도메인(HDFS, Hadoop, OpenStack)의 로그를 통합하여 학습하고 데이터 증강을 적용하는 노트북

## 학습 전략
1. **Domain-Aware Vocabulary**: 각 도메인의 이벤트를 prefix로 구분 (HDFS_E1, Hadoop_E2 등)
2. **Data Augmentation**: 이상 샘플을 증강하여 클래스 불균형 해소
3. **Unified GRU AutoEncoder**: 모든 도메인을 하나의 모델로 학습

## 데이터셋
- HDFS_v1: Hadoop Distributed File System logs
- Hadoop: MapReduce application logs (WordCount, PageRank)
- OpenStack: Cloud platform VM instance logs

## 1. 환경 설정

In [ ]:
# Colab 환경 확인
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running in Google Colab")
except:
    IN_COLAB = False
    print("✅ Running locally")

In [ ]:
# 필수 패키지 설치
!pip install -q torch drain3 regex

In [ ]:
import os
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from collections import Counter, defaultdict
import random
from tqdm import tqdm
import pickle
import json
from datetime import datetime

# Seed 설정
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## 2. 데이터 업로드

Colab에서 실행 시 데이터를 업로드하세요:
1. HDFS_v1/preprocessed/HDFS.npz
2. HDFS_v1/preprocessed/anomaly_label.csv
3. HDFS_v1/preprocessed/HDFS.log_templates.csv
4. Hadoop 폴더 전체 (압축 후 업로드 권장)
5. OpenStack 폴더 전체

In [ ]:
if IN_COLAB:
    from google.colab import files
    print("📤 데이터 파일을 업로드하세요...")
    print("\n옵션 1: 개별 파일 업로드")
    print("  - HDFS.npz, anomaly_label.csv 등")
    print("\n옵션 2: ZIP 파일 업로드 (권장)")
    print("  - 로컬에서 data/test 폴더를 압축하여 업로드")
    print("\n옵션 3: Google Drive 마운트")
    
    # Google Drive 마운트
    from google.colab import drive
    drive.mount('/content/drive')
    
    # 데이터 경로 설정 (Drive에 업로드한 경우)
    BASE_DATA_PATH = "/content/drive/MyDrive/AnomalyDetector/data/test"
    
    # 또는 직접 업로드
    # uploaded = files.upload()
    # BASE_DATA_PATH = "/content"
else:
    # 로컬 경로
    BASE_DATA_PATH = "../data/test"

print(f"✅ Data path: {BASE_DATA_PATH}")

## 3. 데이터셋별 전처리 함수

In [ ]:
class MultiDomainLogDataset:
    """다중 도메인 로그 데이터셋 관리"""
    
    def __init__(self):
        self.datasets = {}  # {domain: {sequences, labels}}
        self.event_vocab = {}  # Unified vocabulary
        self.domain_stats = {}  # Per-domain statistics
        
    def load_hdfs_v1(self, data_path):
        """HDFS_v1 데이터 로드 (이미 전처리된 npz 파일)"""
        print("\n[HDFS_v1] Loading preprocessed data...")
        
        npz_file = os.path.join(data_path, "HDFS_v1/preprocessed/HDFS.npz")
        data = np.load(npz_file, allow_pickle=True)
        
        x_data = data['x_data']  # Event sequences
        y_data = data['y_data']  # Labels (0: normal, 1: anomaly)
        
        # Domain prefix 추가
        sequences = []
        for seq in x_data:
            # 각 이벤트에 HDFS_ prefix 추가
            prefixed_seq = [f"HDFS_{event}" for event in seq]
            sequences.append(prefixed_seq)
        
        self.datasets['hdfs'] = {
            'sequences': sequences,
            'labels': y_data
        }
        
        self.domain_stats['hdfs'] = {
            'total': len(sequences),
            'normal': int((y_data == 0).sum()),
            'anomaly': int((y_data == 1).sum())
        }
        
        print(f"  ✅ Loaded {len(sequences)} sequences")
        print(f"     Normal: {self.domain_stats['hdfs']['normal']}")
        print(f"     Anomaly: {self.domain_stats['hdfs']['anomaly']}")
        
    def load_hadoop(self, data_path):
        """Hadoop 데이터 로드 및 Drain3로 전처리"""
        print("\n[Hadoop] Loading and preprocessing...")
        
        hadoop_path = os.path.join(data_path, "Hadoop")
        
        # Label 파일 읽기
        label_file = os.path.join(hadoop_path, "abnormal_label.txt")
        with open(label_file, 'r') as f:
            label_text = f.read()
        
        # Normal/Abnormal application 파싱
        normal_apps = re.findall(r'Normal:.*?(?=Machine down:|PageRank:)', label_text, re.DOTALL)
        abnormal_apps = re.findall(r'Machine down:.*?(?=PageRank:|$)', label_text, re.DOTALL)
        
        normal_app_ids = []
        for section in normal_apps:
            normal_app_ids.extend(re.findall(r'application_\d+_\d+', section))
        
        abnormal_app_ids = []
        for section in abnormal_apps:
            abnormal_app_ids.extend(re.findall(r'application_\d+_\d+', section))
        
        print(f"  Normal applications: {len(normal_app_ids)}")
        print(f"  Abnormal applications: {len(abnormal_app_ids)}")
        
        # Drain3 초기화
        config = TemplateMinerConfig()
        config.profiling_enabled = False
        miner = TemplateMiner(config=config)
        
        sequences = []
        labels = []
        
        # 각 application 처리
        all_apps = [(app_id, 0) for app_id in normal_app_ids] + \
                   [(app_id, 1) for app_id in abnormal_app_ids]
        
        for app_id, label in tqdm(all_apps, desc="  Processing apps"):
            app_dir = os.path.join(hadoop_path, app_id)
            if not os.path.exists(app_dir):
                continue
            
            # Application 내 모든 로그 파일 읽기
            app_sequence = []
            for log_file in sorted(os.listdir(app_dir)):
                if not log_file.endswith('.log'):
                    continue
                
                log_path = os.path.join(app_dir, log_file)
                try:
                    with open(log_path, 'r', errors='ignore') as f:
                        for line in f:
                            line = line.strip()
                            if not line:
                                continue
                            
                            # Drain3로 템플릿 추출
                            result = miner.add_log_message(line)
                            if result and result['cluster_id']:
                                event_id = f"Hadoop_E{result['cluster_id']}"
                                app_sequence.append(event_id)
                except:
                    continue
            
            if app_sequence:
                sequences.append(app_sequence)
                labels.append(label)
        
        self.datasets['hadoop'] = {
            'sequences': sequences,
            'labels': np.array(labels)
        }
        
        self.domain_stats['hadoop'] = {
            'total': len(sequences),
            'normal': int((np.array(labels) == 0).sum()),
            'anomaly': int((np.array(labels) == 1).sum())
        }
        
        print(f"  ✅ Loaded {len(sequences)} sequences")
        print(f"     Normal: {self.domain_stats['hadoop']['normal']}")
        print(f"     Anomaly: {self.domain_stats['hadoop']['anomaly']}")
    
    def load_openstack(self, data_path):
        """OpenStack 데이터 로드 및 Drain3로 전처리"""
        print("\n[OpenStack] Loading and preprocessing...")
        
        openstack_path = os.path.join(data_path, "OpenStack")
        
        # Anomaly VM IDs 로드
        label_file = os.path.join(openstack_path, "anomaly_labels.txt")
        with open(label_file, 'r') as f:
            lines = f.readlines()
        
        anomaly_vm_ids = set()
        for line in lines:
            line = line.strip()
            # UUID 패턴 추출
            if re.match(r'^[a-f0-9-]{36}$', line):
                anomaly_vm_ids.add(line)
        
        print(f"  Anomaly VM instances: {len(anomaly_vm_ids)}")
        
        # Drain3 초기화
        config = TemplateMinerConfig()
        config.profiling_enabled = False
        miner = TemplateMiner(config=config)
        
        # VM instance별로 로그 그룹핑
        vm_logs = defaultdict(list)
        
        # Normal logs
        for log_file in ['openstack_normal1.log', 'openstack_normal2.log']:
            log_path = os.path.join(openstack_path, log_file)
            if os.path.exists(log_path):
                with open(log_path, 'r', errors='ignore') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        
                        # VM ID 추출 (UUID 패턴)
                        vm_match = re.search(r'([a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12})', line)
                        if vm_match:
                            vm_id = vm_match.group(1)
                            vm_logs[vm_id].append(line)
        
        # Abnormal logs
        log_path = os.path.join(openstack_path, 'openstack_abnormal.log')
        if os.path.exists(log_path):
            with open(log_path, 'r', errors='ignore') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    
                    vm_match = re.search(r'([a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12})', line)
                    if vm_match:
                        vm_id = vm_match.group(1)
                        vm_logs[vm_id].append(line)
        
        print(f"  Total VM instances: {len(vm_logs)}")
        
        # VM별 시퀀스 생성
        sequences = []
        labels = []
        
        for vm_id, logs in tqdm(vm_logs.items(), desc="  Processing VMs"):
            if len(logs) < 5:  # 너무 짧은 시퀀스 제외
                continue
            
            vm_sequence = []
            for log_line in logs:
                result = miner.add_log_message(log_line)
                if result and result['cluster_id']:
                    event_id = f"OpenStack_E{result['cluster_id']}"
                    vm_sequence.append(event_id)
            
            if vm_sequence:
                sequences.append(vm_sequence)
                # VM이 anomaly list에 있으면 1, 아니면 0
                label = 1 if vm_id in anomaly_vm_ids else 0
                labels.append(label)
        
        self.datasets['openstack'] = {
            'sequences': sequences,
            'labels': np.array(labels)
        }
        
        self.domain_stats['openstack'] = {
            'total': len(sequences),
            'normal': int((np.array(labels) == 0).sum()),
            'anomaly': int((np.array(labels) == 1).sum())
        }
        
        print(f"  ✅ Loaded {len(sequences)} sequences")
        print(f"     Normal: {self.domain_stats['openstack']['normal']}")
        print(f"     Anomaly: {self.domain_stats['openstack']['anomaly']}")
    
    def build_vocabulary(self):
        """통합 vocabulary 생성"""
        print("\n[Vocabulary] Building unified vocabulary...")
        
        all_events = set()
        for domain, data in self.datasets.items():
            for seq in data['sequences']:
                all_events.update(seq)
        
        # PAD와 UNK 토큰 추가
        self.event_vocab = {'<PAD>': 0, '<UNK>': 1}
        for idx, event in enumerate(sorted(all_events), start=2):
            self.event_vocab[event] = idx
        
        print(f"  ✅ Vocabulary size: {len(self.event_vocab)}")
        
        # 도메인별 이벤트 수
        for domain in self.datasets.keys():
            domain_events = set()
            for seq in self.datasets[domain]['sequences']:
                domain_events.update(seq)
            print(f"     {domain}: {len(domain_events)} unique events")
    
    def print_statistics(self):
        """전체 통계 출력"""
        print("\n" + "=" * 80)
        print("Dataset Statistics")
        print("=" * 80)
        
        total_samples = 0
        total_normal = 0
        total_anomaly = 0
        
        for domain, stats in self.domain_stats.items():
            print(f"\n{domain.upper()}:")
            print(f"  Total: {stats['total']}")
            print(f"  Normal: {stats['normal']} ({stats['normal']/stats['total']*100:.2f}%)")
            print(f"  Anomaly: {stats['anomaly']} ({stats['anomaly']/stats['total']*100:.2f}%)")
            
            total_samples += stats['total']
            total_normal += stats['normal']
            total_anomaly += stats['anomaly']
        
        print(f"\nOVERALL:")
        print(f"  Total: {total_samples}")
        print(f"  Normal: {total_normal} ({total_normal/total_samples*100:.2f}%)")
        print(f"  Anomaly: {total_anomaly} ({total_anomaly/total_samples*100:.2f}%)")
        print(f"  Imbalance ratio: {total_normal/total_anomaly:.1f}:1")
        print("=" * 80)

## 4. 데이터 로드

In [ ]:
# 데이터셋 로드
dataset = MultiDomainLogDataset()

# HDFS
dataset.load_hdfs_v1(BASE_DATA_PATH)

# Hadoop
dataset.load_hadoop(BASE_DATA_PATH)

# OpenStack
dataset.load_openstack(BASE_DATA_PATH)

# Vocabulary 생성
dataset.build_vocabulary()

# 통계 출력
dataset.print_statistics()

## 5. 데이터 증강

In [ ]:
class DataAugmenter:
    """로그 시퀀스 데이터 증강"""
    
    def __init__(self, event_vocab):
        self.event_vocab = event_vocab
        self.vocab_list = list(event_vocab.keys())
        
    def add_noise(self, sequence, noise_ratio=0.1):
        """시퀀스에 노이즈 추가"""
        seq = sequence.copy()
        num_changes = max(1, int(len(seq) * noise_ratio))
        indices = random.sample(range(len(seq)), min(num_changes, len(seq)))
        
        for idx in indices:
            seq[idx] = random.choice(self.vocab_list[2:])  # PAD, UNK 제외
        
        return seq
    
    def permute(self, sequence, num_swaps=2):
        """시퀀스 순서 변경"""
        seq = sequence.copy()
        
        for _ in range(num_swaps):
            if len(seq) >= 2:
                i, j = random.sample(range(len(seq)), 2)
                seq[i], seq[j] = seq[j], seq[i]
        
        return seq
    
    def truncate(self, sequence, min_ratio=0.7):
        """시퀀스 자르기"""
        seq = sequence.copy()
        new_length = max(1, int(len(seq) * random.uniform(min_ratio, 1.0)))
        start_idx = random.randint(0, len(seq) - new_length)
        
        return seq[start_idx:start_idx + new_length]
    
    def duplicate(self, sequence, dup_ratio=0.2):
        """이벤트 중복"""
        seq = sequence.copy()
        num_dups = max(1, int(len(seq) * dup_ratio))
        
        for _ in range(num_dups):
            idx = random.randint(0, len(seq) - 1)
            seq.insert(idx, seq[idx])
        
        return seq
    
    def substitute(self, sequence, sub_ratio=0.15):
        """유사 이벤트로 치환"""
        seq = sequence.copy()
        num_subs = max(1, int(len(seq) * sub_ratio))
        indices = random.sample(range(len(seq)), min(num_subs, len(seq)))
        
        for idx in indices:
            # 같은 도메인의 이벤트로 치환
            current_event = seq[idx]
            if '_' in current_event:
                domain_prefix = current_event.split('_')[0]
                domain_events = [e for e in self.vocab_list if e.startswith(domain_prefix + '_')]
                if domain_events:
                    seq[idx] = random.choice(domain_events)
        
        return seq
    
    def augment(self, sequences, labels, augmentation_factor=3):
        """이상 샘플 증강"""
        print(f"\n[Augmentation] Augmenting anomaly samples...")
        print(f"  Target factor: {augmentation_factor}x")
        
        # 정상/이상 분리
        normal_seqs = [seq for seq, label in zip(sequences, labels) if label == 0]
        anomaly_seqs = [seq for seq, label in zip(sequences, labels) if label == 1]
        
        print(f"  Original - Normal: {len(normal_seqs)}, Anomaly: {len(anomaly_seqs)}")
        
        # 증강 메서드
        methods = [
            ('Noise', self.add_noise),
            ('Permute', self.permute),
            ('Truncate', self.truncate),
            ('Duplicate', self.duplicate),
            ('Substitute', self.substitute)
        ]
        
        # 증강된 이상 샘플
        augmented_anomaly = anomaly_seqs.copy()
        
        num_to_generate = len(anomaly_seqs) * (augmentation_factor - 1)
        num_per_method = num_to_generate // len(methods)
        
        for method_name, method_func in methods:
            print(f"  Applying {method_name}... ", end='')
            
            for _ in range(num_per_method):
                original = random.choice(anomaly_seqs)
                augmented = method_func(original)
                augmented_anomaly.append(augmented)
            
            print(f"✅ {num_per_method} samples")
        
        # 합치기
        all_sequences = normal_seqs + augmented_anomaly
        all_labels = [0] * len(normal_seqs) + [1] * len(augmented_anomaly)
        
        # 셔플
        combined = list(zip(all_sequences, all_labels))
        random.shuffle(combined)
        all_sequences, all_labels = zip(*combined)
        
        print(f"\n  ✅ Augmentation complete!")
        print(f"     Total: {len(all_sequences)}")
        print(f"     Normal: {sum(1 for l in all_labels if l == 0)}")
        print(f"     Anomaly: {sum(1 for l in all_labels if l == 1)}")
        print(f"     New ratio: {sum(1 for l in all_labels if l == 0) / sum(1 for l in all_labels if l == 1):.1f}:1")
        
        return list(all_sequences), list(all_labels)

In [ ]:
# 모든 도메인 데이터 합치기
all_sequences = []
all_labels = []

for domain, data in dataset.datasets.items():
    all_sequences.extend(data['sequences'])
    all_labels.extend(data['labels'])

print(f"Combined dataset: {len(all_sequences)} sequences")

# 데이터 증강
augmenter = DataAugmenter(dataset.event_vocab)
augmented_sequences, augmented_labels = augmenter.augment(
    all_sequences, 
    all_labels, 
    augmentation_factor=3
)

## 6. 시퀀스를 숫자로 변환 및 데이터셋 생성

In [ ]:
class LogSequenceDataset(Dataset):
    """PyTorch Dataset for log sequences"""
    
    def __init__(self, sequences, labels, event_vocab, max_len=50):
        self.sequences = sequences
        self.labels = labels
        self.event_vocab = event_vocab
        self.max_len = max_len
        self.num_events = len(event_vocab)
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        label = self.labels[idx]
        
        # 이벤트 ID를 인덱스로 변환
        event_indices = [self.event_vocab.get(event, 1) for event in sequence]  # 1 = UNK
        
        # Padding/Truncate
        if len(event_indices) > self.max_len:
            event_indices = event_indices[:self.max_len]
        else:
            event_indices = event_indices + [0] * (self.max_len - len(event_indices))  # 0 = PAD
        
        # One-hot encoding
        one_hot = np.zeros((self.max_len, self.num_events), dtype=np.float32)
        for i, event_idx in enumerate(event_indices):
            one_hot[i, event_idx] = 1.0
        
        return torch.FloatTensor(one_hot), torch.FloatTensor([label])

# Train/Val/Test 분할
print("\n[Dataset] Splitting data...")

# 80% train, 10% val, 10% test
n = len(augmented_sequences)
indices = list(range(n))
random.shuffle(indices)

train_size = int(0.8 * n)
val_size = int(0.1 * n)

train_idx = indices[:train_size]
val_idx = indices[train_size:train_size + val_size]
test_idx = indices[train_size + val_size:]

train_sequences = [augmented_sequences[i] for i in train_idx]
train_labels = [augmented_labels[i] for i in train_idx]

val_sequences = [augmented_sequences[i] for i in val_idx]
val_labels = [augmented_labels[i] for i in val_idx]

test_sequences = [augmented_sequences[i] for i in test_idx]
test_labels = [augmented_labels[i] for i in test_idx]

print(f"  Train: {len(train_sequences)}")
print(f"  Val: {len(val_sequences)}")
print(f"  Test: {len(test_sequences)}")

# 최대 시퀀스 길이 계산
max_seq_len = max(len(seq) for seq in augmented_sequences)
print(f"  Max sequence length: {max_seq_len}")

# 50으로 제한
MAX_LEN = min(50, max_seq_len)
print(f"  Using max length: {MAX_LEN}")

# Dataset 생성
train_dataset = LogSequenceDataset(train_sequences, train_labels, dataset.event_vocab, MAX_LEN)
val_dataset = LogSequenceDataset(val_sequences, val_labels, dataset.event_vocab, MAX_LEN)
test_dataset = LogSequenceDataset(test_sequences, test_labels, dataset.event_vocab, MAX_LEN)

# DataLoader
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n  ✅ DataLoaders created (batch_size={BATCH_SIZE})")

## 7. GRU AutoEncoder 모델 정의

In [ ]:
class GRUAutoEncoder(nn.Module):
    """GRU 기반 시퀀스 AutoEncoder"""
    
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.1):
        super(GRUAutoEncoder, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Encoder
        self.encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        # Decoder
        self.decoder = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        # Output layer
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # Encode
        encoded, h_n = self.encoder(x)
        
        # Decode
        last_hidden = h_n[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        decoded, _ = self.decoder(last_hidden)
        
        # Output
        output = self.output_layer(decoded)
        
        return output

# 모델 초기화
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n[Model] Using device: {device}")

NUM_EVENTS = len(dataset.event_vocab)
HIDDEN_DIM = 128
NUM_LAYERS = 2

model = GRUAutoEncoder(
    input_dim=NUM_EVENTS,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS
).to(device)

print(f"  Input dim: {NUM_EVENTS}")
print(f"  Hidden dim: {HIDDEN_DIM}")
print(f"  Num layers: {NUM_LAYERS}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 8. 학습

In [ ]:
# 학습 설정
NUM_EPOCHS = 50
LEARNING_RATE = 0.001
PATIENCE = 5  # Early stopping

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"\n[Training] Starting training...")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Early stopping patience: {PATIENCE}")

best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0
train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    
    for batch_x, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False):
        batch_x = batch_x.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_x)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for batch_x, _ in val_loader:
            batch_x = batch_x.to(device)
            output = model(batch_x)
            loss = criterion(output, batch_x)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        patience_counter = 0
        # 모델 저장
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  ✅ New best model saved (val_loss: {val_loss:.6f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⚠️  Early stopping at epoch {epoch+1}")
            break

print(f"\n✅ Training complete!")
print(f"  Best validation loss: {best_val_loss:.6f} at epoch {best_epoch}")

## 9. Threshold 계산

In [ ]:
# Best 모델 로드
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

print("\n[Threshold] Calculating threshold from validation set...")

val_losses = []

with torch.no_grad():
    for batch_x, _ in val_loader:
        batch_x = batch_x.to(device)
        output = model(batch_x)
        
        # Sample별 loss 계산
        for i in range(batch_x.size(0)):
            sample_loss = criterion(output[i], batch_x[i]).item()
            val_losses.append(sample_loss)

val_losses = np.array(val_losses)

# Conservative threshold (95 percentile)
threshold_p95 = np.percentile(val_losses, 95)
threshold_mean_std = np.mean(val_losses) + 3 * np.std(val_losses)
threshold = max(threshold_p95, threshold_mean_std)

print(f"  Validation loss stats:")
print(f"    Mean: {np.mean(val_losses):.6f}")
print(f"    Std: {np.std(val_losses):.6f}")
print(f"    Min: {np.min(val_losses):.6f}")
print(f"    Max: {np.max(val_losses):.6f}")
print(f"    95th percentile: {threshold_p95:.6f}")
print(f"    Mean + 3*Std: {threshold_mean_std:.6f}")
print(f"\n  ✅ Selected threshold: {threshold:.6f}")

## 10. 테스트 세트 평가

In [ ]:
print("\n[Evaluation] Testing on test set...")

predictions = []
true_labels = []
test_losses = []

model.eval()
with torch.no_grad():
    for batch_x, batch_y in tqdm(test_loader, desc="Testing"):
        batch_x = batch_x.to(device)
        output = model(batch_x)
        
        # Sample별 loss 계산
        for i in range(batch_x.size(0)):
            sample_loss = criterion(output[i], batch_x[i]).item()
            test_losses.append(sample_loss)
            
            # Threshold 기반 예측
            pred = 1 if sample_loss > threshold else 0
            predictions.append(pred)
            true_labels.append(int(batch_y[i].item()))

predictions = np.array(predictions)
true_labels = np.array(true_labels)

# Confusion Matrix
TP = np.sum((predictions == 1) & (true_labels == 1))
TN = np.sum((predictions == 0) & (true_labels == 0))
FP = np.sum((predictions == 1) & (true_labels == 0))
FN = np.sum((predictions == 0) & (true_labels == 1))

# Metrics
accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "=" * 80)
print("Test Set Evaluation Results")
print("=" * 80)

print(f"\nConfusion Matrix:")
print(f"  True Positives (TP):  {TP:5d}  (Correctly detected anomalies)")
print(f"  True Negatives (TN):  {TN:5d}  (Correctly detected normal)")
print(f"  False Positives (FP): {FP:5d}  (Normal misclassified as anomaly)")
print(f"  False Negatives (FN): {FN:5d}  (Anomaly misclassified as normal)")

print(f"\nMetrics:")
print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"  Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"  F1 Score:  {f1_score:.4f}")

print("=" * 80)

if accuracy > 0.9:
    print("\n✅ Excellent performance!")
elif accuracy > 0.8:
    print("\n✅ Good performance!")
elif accuracy > 0.7:
    print("\n⚠️  Moderate performance. Consider tuning hyperparameters.")
else:
    print("\n⚠️  Low performance. Consider:")
    print("   - Adjusting threshold")
    print("   - More augmentation")
    print("   - Different model architecture")

## 11. 모델 저장

In [ ]:
# 모델 메타데이터
metadata = {
    'model_type': 'multi_domain_gru_autoencoder',
    'domains': list(dataset.datasets.keys()),
    'num_events': NUM_EVENTS,
    'input_dim': NUM_EVENTS,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'max_seq_len': MAX_LEN,
    'threshold': threshold,
    'best_val_loss': best_val_loss,
    'num_epochs': best_epoch,
    'training_date': datetime.now().strftime('%Y%m%d_%H%M%S'),
    'train_size': len(train_sequences),
    'val_size': len(val_sequences),
    'test_size': len(test_sequences),
    'test_accuracy': accuracy,
    'test_precision': precision,
    'test_recall': recall,
    'test_f1': f1_score
}

# 저장
print("\n[Save] Saving model and metadata...")

# 모델
torch.save(model.state_dict(), 'multi_domain_gru_autoencoder.pth')
print("  ✅ Model saved: multi_domain_gru_autoencoder.pth")

# Vocabulary
with open('multi_domain_event_vocab.pkl', 'wb') as f:
    pickle.dump(dataset.event_vocab, f)
print("  ✅ Vocabulary saved: multi_domain_event_vocab.pkl")

# Threshold
with open('multi_domain_threshold.pkl', 'wb') as f:
    pickle.dump(threshold, f)
print("  ✅ Threshold saved: multi_domain_threshold.pkl")

# Metadata
with open('multi_domain_model_meta.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print("  ✅ Metadata saved: multi_domain_model_meta.pkl")

# JSON으로도 저장
with open('multi_domain_model_meta.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("  ✅ Metadata saved: multi_domain_model_meta.json")

print("\n✅ All files saved successfully!")
print("\nDownload these files to your local machine:")
print("  - multi_domain_gru_autoencoder.pth")
print("  - multi_domain_event_vocab.pkl")
print("  - multi_domain_threshold.pkl")
print("  - multi_domain_model_meta.pkl")

## 12. Colab에서 파일 다운로드

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    print("다운로드 중...")
    files.download('multi_domain_gru_autoencoder.pth')
    files.download('multi_domain_event_vocab.pkl')
    files.download('multi_domain_threshold.pkl')
    files.download('multi_domain_model_meta.pkl')
    files.download('multi_domain_model_meta.json')
    print("✅ 다운로드 완료!")
else:
    print("로컬 환경에서는 파일이 현재 디렉토리에 저장되었습니다.")